In [1]:
import requests
from bs4 import BeautifulSoup
import time
import os

# --- Step 1: Automatically extract links ---
main_url = "https://www.flowermeaning.com/"
try:
    print(f"Fetching article links from {main_url}...")
    main_response = requests.get(main_url)
    main_soup = BeautifulSoup(main_response.text, "html.parser")
    urls = [a["href"] for a in main_soup.select("div.fusion-megamenu-title a") if a.get("href")]
    urls = list(dict.fromkeys(urls)) # Remove duplicates
    print(f"Found {len(urls)} flower links.")
except Exception as e:
    print(f"Error fetching main list: {e}")
    urls = []

# --- Step 2: Scrape and Clean ---
output_dir = "flower_texts"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for url in urls:
    flower_name = url.split('/')[-2]
    filename = f"{output_dir}/{flower_name}.txt"
    print(f"Processing: {flower_name}...")

    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        content = soup.find("div", class_="post-content")

        if content:
            # 1. Remove Social Sharing
            for social in content.find_all(class_=["essb_links", "essb_displayed_postfloat"]):
                social.decompose()

            # 2. Remove known ad containers (code-blocks)
            for ad_box in content.find_all("div", class_="code-block"):
                ad_box.decompose()

            # 3. Robust removal: Find any element containing known ad keywords
            ad_keywords = ["Click here"]
            for keyword in ad_keywords:
                for element in content.find_all(string=lambda text: keyword in text if text else False):
                    # Find the parent container (like a div or p) and remove it entirely
                    parent = element.find_parent(["div", "p", "section"])
                    if parent:
                        parent.decompose()

            # Clean text extraction
            text = content.get_text(separator="\n", strip=True)

            with open(filename, "w", encoding="utf-8") as f:
                f.write(text)

        time.sleep(1)
    except Exception as e:
        print(f"Error processing {url}: {e}")

print("\nScraping finished.")

Fetching article links from https://www.flowermeaning.com/...
Found 98 flower links.
Processing: alstroemeria-flower-meaning...
Processing: amaryllis-flower-meaning...
Processing: anemone-flower-meaning...
Processing: anthurium-flower...
Processing: aster-flower-meaning...
Processing: azalea-flower-meaning...
Processing: baby-breath-flower...
Processing: begonia-flower...
Processing: bird-of-paradise-flower-meaning...
Processing: bleeding-heart-flower-meaning...
Processing: buttercup-flower-meaning...
Processing: cactus-flower-meaning...
Processing: calla-lily-meaning...
Processing: camellia-flower-meaning...
Processing: carnation-flower-meaning...
Processing: christmas-flowers...
Processing: chrysanthemum-flower-meaning...
Processing: columbine-flower-meaning...
Processing: crocus-flower-meaning...
Processing: daffodil-flower-meaning...
Processing: dahlia-flower-meaning...
Processing: daisy-flower-meaning...
Processing: dandelion-flower-meaning...
Processing: delphinium...
Processing: